In [ ]:
# ============================================================
# RF SIGNAL CLASSIFICATION USING CNN
# RadioML 2016.10A Dataset
# ============================================================

# ==========================
# IMPORT LIBRARIES
# ==========================

import pickle
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    Flatten,
    Dense,
    Dropout
)

# ==========================
# LOAD DATASET
# ==========================

DATASET_PATH = "RML2016.10a_dict.pkl"

print("Loading Dataset...")

with open(DATASET_PATH, 'rb') as f:
    Xd = pickle.load(f, encoding='latin1')

mods = sorted(list(set(k[0] for k in Xd.keys())))
snrs = sorted(list(set(k[1] for k in Xd.keys())))

X = []
Y = []

for mod in mods:
    for snr in snrs:

        samples = Xd[(mod, snr)]

        X.append(samples)

        Y.extend([mod] * samples.shape[0])

X = np.vstack(X)

print("\nDataset Loaded Successfully")
print("Total Samples :", X.shape[0])
print("Sample Shape  :", X.shape)
print("Classes       :", mods)

# ==========================
# IQ WAVEFORM VISUALIZATION
# ==========================

sample_index = 50000

signal = X[sample_index]

I = signal[0]
Q = signal[1]

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(I)
plt.title("In-Phase (I)")

plt.subplot(1,2,2)
plt.plot(Q)
plt.title("Quadrature (Q)")

plt.tight_layout()

plt.savefig(
    "iq_waveform.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# ==========================
# CONSTELLATION DIAGRAM
# ==========================

plt.figure(figsize=(6,6))

plt.scatter(I, Q, s=10)

plt.title(
    f"Constellation : {Y[sample_index]}"
)

plt.xlabel("I")
plt.ylabel("Q")

plt.grid(True)

plt.savefig(
    "constellation_8psk.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# ==========================
# PREPROCESSING
# ==========================

print("\nPreprocessing Dataset...")

X = np.transpose(X, (0,2,1))

encoder = LabelEncoder()

Y_encoded = encoder.fit_transform(Y)

Y_cat = to_categorical(Y_encoded)

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y_cat,
    test_size=0.2,
    random_state=42
)

print("\nTraining Samples :", X_train.shape)
print("Testing Samples  :", X_test.shape)

# ==========================
# CNN MODEL
# ==========================

model = Sequential()

model.add(
    Conv1D(
        filters=64,
        kernel_size=3,
        activation='relu',
        input_shape=X_train.shape[1:]
    )
)

model.add(
    Conv1D(
        filters=128,
        kernel_size=3,
        activation='relu'
    )
)

model.add(Flatten())

model.add(
    Dense(
        256,
        activation='relu'
    )
)

model.add(
    Dropout(0.5)
)

model.add(
    Dense(
        len(mods),
        activation='softmax'
    )
)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nModel Summary\n")

model.summary()

# ==========================
# TRAIN MODEL
# ==========================

print("\nStarting Training...\n")

history = model.fit(
    X_train,
    Y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.1
)

# ==========================
# EVALUATE MODEL
# ==========================

print("\nEvaluating Model...\n")

loss, accuracy = model.evaluate(
    X_test,
    Y_test,
    verbose=1
)

print("\n====================")
print("Test Accuracy :", accuracy)
print("Test Loss     :", loss)
print("====================")

# ==========================
# SAVE MODEL
# ==========================

model.save(
    "rf_classifier.keras"
)

print(
    "\nModel Saved Successfully"
)

# ==========================
# ACCURACY CURVE
# ==========================

plt.figure(figsize=(8,5))

plt.plot(
    history.history['accuracy'],
    linewidth=2
)

plt.plot(
    history.history['val_accuracy'],
    linewidth=2
)

plt.title(
    'CNN Accuracy Curve'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.legend([
    'Train Accuracy',
    'Validation Accuracy'
])

plt.grid(True)

plt.savefig(
    'accuracy_curve.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# ==========================
# LOSS CURVE
# ==========================

plt.figure(figsize=(8,5))

plt.plot(
    history.history['loss'],
    linewidth=2
)

plt.plot(
    history.history['val_loss'],
    linewidth=2
)

plt.title(
    'CNN Loss Curve'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.legend([
    'Train Loss',
    'Validation Loss'
])

plt.grid(True)

plt.savefig(
    'loss_curve.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# ==========================
# FINAL OUTPUT
# ==========================

print("\nGenerated Files:")

print("1. iq_waveform.png")
print("2. constellation_8psk.png")
print("3. accuracy_curve.png")
print("4. loss_curve.png")
print("5. rf_classifier.keras")

print("\nProject Completed Successfully!")